In [ ]:
import xml.etree.ElementTree as et
import pandas as pd
import re
import os
from iso3166 import countries
import src.xml_parse as parse_utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
DATA_PATH = "/gpfs01/berens/data/data/pubmed_central/baseline_2025-06-26"
SECTION = "PMC008xxxxxx_noncomm"
FILENAME = "PMC8551494.xml"

In [ ]:
xml_file = os.path.join(DATA_PATH, SECTION, FILENAME)
xtree = et.parse(xml_file, parser=et.XMLParser(encoding="UTF-8"))
xroot = xtree.getroot()

In [ ]:
for child in xroot:
    print(child.tag, child.attrib)
    for child2 in child:
        print("    ", child2.tag, child2.attrib)
        for child3 in child2:
            print("        ", child3.tag, child3.attrib)

In [ ]:
xroot.tag

In [ ]:
body = xroot.findall("./body")[0]

In [ ]:
secs = xroot.findall("./body/sec")
len(secs)

In [ ]:
sections_dict = {}
for sec in secs:
    text = " ".join(sec.itertext())
    for child in sec:
        # only parse sections with title, to avoid mismatch
        if child.tag == "title":
            title = child.text
            sections_dict[title] = text

sections_dict

In [ ]:
# if there are more sections than section titles, and the first section
# is untitled, it is assumed to be the introduction
first_sec = " ".join(secs[0].itertext())
first_sec_in_dict = list(sections_dict.values())[0]
if len(sections_dict) < len(secs):
    if not first_sec == first_sec_in_dict:
        sections_dict["Introduction"] = first_sec

sections_dict

In [ ]:
# get the first text of the body (which is not inside of a section)
# if this exists, it is assumed to be the introduction
# this can cause problems, e.g. in "PMC8483461.xml", where this part is
# not the introduction, but the abbreviations and highlights
# in other papers, this includes the introduction, but also other
# such as key insights
intro = ""
for child in body:
    if child.tag == "sec":
        break
    intro += " ".join(child.itertext())
if len(intro) > 0:
    sections_dict["Introduction"] = intro

sections_dict

In [ ]:
list(sections_dict.keys())

In [ ]:
list(sections_dict.values())

In [ ]:
parse_utils.xml_get_text([secs[0]])

In [ ]:
titles = xroot.findall("./body/sec/title")
len(titles)

In [ ]:
title_names = parse_utils.get_sec_titles(xroot)
title_names

In [ ]:
parse_utils.xml_get_attr(xroot, "{http://www.w3.org/XML/1998/namespace}lang")

In [ ]:
parse_utils.xml_get_attr(xroot, "article-type")

In [ ]:
parse_utils.xml_get_text(
    xroot.findall("./front/journal-meta/journal-title-group/journal-title")
)

In [ ]:
# not all articles have pmid, so also collect pmc id
parse_utils.xml_get_text(
    xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmid"]')
)

In [ ]:
parse_utils.xml_get_text(
    xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmc"]')
)

In [ ]:
parse_utils.xml_get_text(
    xroot.findall("./front/article-meta/title-group/article-title")
)

In [ ]:
parse_utils.get_country(xroot)

In [ ]:
parse_utils.get_date(xroot)

In [ ]:
# alternative version: get date dict to include tags to make sure day and month
# aren't confused
nodes = xroot.findall('./front/article-meta/pub-date/[@pub-type="epub"]')
date = {}
if len(nodes) > 0:
    for child in nodes[0]:
        date[child.tag] = child.text
date

In [ ]:
parse_utils.get_abstr(xroot.findall("./front/article-meta/abstract"))

In [ ]:
parse_utils.xml_parse_single(xml_file)